<a href="https://colab.research.google.com/github/LuisTT0903/Processamento-de-Sinais1/blob/main/Pratica3Quest%C3%A3o6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import urllib.request
from scipy.io import wavfile

url = "https://raw.githubusercontent.com/LuisTT0903/Processamento-de-Sinais1/main/Aula_03/Audios_Usados/handel.wav"
urllib.request.urlretrieve(url, "handel.wav")
fs, x_int = wavfile.read("handel.wav")
x = x_int.astype(np.float64) / np.iinfo(x_int.dtype).max

def coeficientes(a, L):
    b = np.zeros(L+1); b[0]=1; b[L]=-1
    a_coef = np.zeros(L+1); a_coef[0]=1; a_coef[L]=-a
    return b, a_coef

def inverso_exato(a, L):
    b, a_coef = coeficientes(a, L)
    return a_coef.copy(), b.copy()   # G(z) = 1/H(z): numerador e denominador trocados

valores_a = [0.7, 0.9]; valores_L = [1, 4, 10]

# checa que os polos de G ficam sempre exatamente sobre o círculo unitário
for a in valores_a:
    for L in valores_L:
        b_inv, a_inv = inverso_exato(a, L)
        _, polos, _ = signal.tf2zpk(b_inv, a_inv)
        print(f"a={a}, L={L}: maior |polo| de G = {np.max(np.abs(polos)):.6f}")

# resposta em frequência e diagrama de polos/zeros de G, para as 6 combinações
# (eixo y limitado no gráfico de módulo, pois o ganho tende a infinito nas frequências dos polos)
...

# recuperação: cenário ideal (float, sem ruído) vs. realista (a partir do .wav salvo em 16 bits)
for a in valores_a:
    for L in valores_L:
        b, a_coef = coeficientes(a, L)
        b_inv, a_inv = inverso_exato(a, L)
        y = signal.lfilter(b, a_coef, x)

        x_rec_ideal = signal.lfilter(b_inv, a_inv, y)
        SNR_ideal = 10*np.log10(np.sum(x**2)/np.sum((x-x_rec_ideal)**2))

        ganho = np.max(np.abs(y)); y_norm = y/ganho
        wavfile.write("tmp.wav", fs, (y_norm*32767).astype(np.int16))
        _, y_int = wavfile.read("tmp.wav")
        y_salvo = (y_int.astype(np.float64)/32767)*ganho
        x_rec_real = signal.lfilter(b_inv, a_inv, y_salvo)
        SNR_real = 10*np.log10(np.sum(x**2)/np.sum((x-x_rec_real)**2))

        print(a, L, SNR_ideal, SNR_real)

a=0.7, L=1: maior |polo| de G = 1.000000
a=0.7, L=4: maior |polo| de G = 1.000000
a=0.7, L=10: maior |polo| de G = 1.000000
a=0.9, L=1: maior |polo| de G = 1.000000
a=0.9, L=4: maior |polo| de G = 1.000000
a=0.9, L=10: maior |polo| de G = 1.000000
0.7 1 278.12257938955406 48.89740433071849
0.7 4 284.55112262609845 51.065186736592956
0.7 10 286.86192412973946 58.18642316422675
0.9 1 274.80951022930805 50.85323250077746
0.9 4 288.8636578231725 61.03977898079802
0.9 10 288.33419138593274 67.72401698418383


O inverso exato
𝐺
(
𝑧
)
G(z) tem todos os seus polos exatamente sobre o círculo unitário (confirmado numérica e graficamente), porque os zeros de
𝐻
(
𝑧
)
H(z) nessa questão (as raízes
𝐿
L-ésimas da unidade) já estavam sobre o círculo unitário na Questão 4 — diferente da Questão 1/3, onde ficavam estritamente dentro. Por isso,
𝐺
(
𝑧
)
G(z) é apenas marginalmente estável, e sua resposta em frequência tende a
+
∞
+∞ exatamente nas frequências que
𝐻
(
𝑧
)
H(z) havia zerado (não existe ganho finito capaz de recuperar uma informação exatamente anulada).

No cenário ideal (aplicando
𝐺
(
𝑧
)
G(z) direto a
𝑦
[
𝑛
]
y[n] em ponto flutuante, sem ruído), a recuperação é praticamente perfeita (SNR > 270 dB em todos os casos), porque
𝐺
(
𝑧
)
𝐻
(
𝑧
)
=
1
G(z)H(z)=1 é uma identidade algébrica exata para sequências causais — a marginalidade da estabilidade não atrapalha aqui, já que não há nada para os polos "integrarem".

No cenário realista (recuperando a partir do áudio salvo em 16 bits, com ruído de quantização), a SNR cai para 49–68 dB — visivelmente pior que os ~79 dB da Questão 3 nas mesmas condições. Mais revelador: o erro de recuperação, medido por trecho do áudio, tende a crescer ao longo da gravação em vez de ficar estacionário — sintoma direto de que os polos sobre o círculo unitário integram o ruído de quantização sem deixá-lo decair, diferente da Questão 3 (onde o erro ficava limitado do início ao fim).

Uma alternativa mais robusta é puxar o polo do inverso para um raio levemente menor que 1 (testei com r=0,995): isso garante estabilidade estrita e erro que não cresce com a duração do sinal, ao custo de uma SNR global um pouco pior (introduz um viés constante de aproximação, em vez de um erro que só se acumula aos poucos) — mais confiável para sinais mais longos ou mais ruidosos que os ~9 segundos testados aqui.

Em suma: diferente da Questão 3, aqui a recuperação funciona muito bem apenas na teoria/em ponto flutuante, mas é estruturalmente frágil diante de qualquer imperfeição real do sinal, porque os zeros de
𝐻
(
𝑧
)
H(z) representam perda de informação genuinamente irrecuperável naquelas frequências exatas.